## Imports

In [ ]:
import os

from alphagenome.data import genome
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
from alphagenome.models.v1 import variant_scorers
from alphagenome.models.v1 import dna_client
import matplotlib.pyplot as plt
import pandas as pd


## Local Model Alpha Genome

Use the `alphagenome_biowulf` environment with the versions pinned in `pyproject.toml` and `uv.lock`. Restart the kernel after upgrading and rerun from Imports; an already loaded model retains the old implementation.

Local quantile scoring requires the official human calibration data. The setup below downloads it once over HTTPS into `~/.cache/alphagenome/calibration` and reuses it, honoring the HTTP(S) proxy environment on Biowulf. No hosted inference API key is required. Default human annotations and calibration are preserved alongside the local GRCh38 FASTA. Quantiles describe the reference score distribution, not disease causality or experimental validation. Their scale depends on the scorer: RNA_SEQ quantiles can be negative, unlike a conventional percentile bounded by 0 and 1.

In [ ]:
# Load model using persistent HF auth and local FASTA override
import dataclasses
import os
from pathlib import Path
from getpass import getpass
from huggingface_hub import whoami, login
import jax
from alphagenome_research.model import dna_model
from alphagenome.models.v1 import dna_model as public_dna_model
import fsspec


gpu_devices = [device for device in jax.local_devices() if device.platform == "gpu"]
if not gpu_devices:
    raise RuntimeError(
        "JAX cannot see a GPU in this notebook kernel. Restart the kernel and "
        "rerun this cell using the alphagenome_biowulf environment."
    )


def ensure_hf_auth(allow_prompt=True):
    try:
        info = whoami()
        print(f"HF already authenticated as: {info.get('name', 'unknown')}")
        return
    except Exception:
        pass

    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if token:
        login(token=token, add_to_git_credential=False, skip_if_logged_in=True)
        info = whoami()
        print(f"HF authenticated via environment token as: {info.get('name', 'unknown')}")
        return

    if allow_prompt:
        try:
            token = getpass("Enter Hugging Face token (input hidden): ").strip()
        except Exception:
            token = ""

        if token:
            login(token=token, add_to_git_credential=False, skip_if_logged_in=True)
            info = whoami()
            print(f"HF authenticated via interactive prompt as: {info.get('name', 'unknown')}")
            return

    raise RuntimeError(
        "No Hugging Face credentials found. Use one of:\n"
        "1) Run `hf auth login` in a terminal\n"
        "2) Set HF_TOKEN (or HUGGINGFACEHUB_API_TOKEN)\n"
        "3) Re-run this cell and paste a token when prompted"
    )


def resolve_local_human_fasta_path():
    candidates = [
        os.getenv("ALPHAGENOME_FASTA"),
        "/data/Dcode/gaetano/repos/fasta_files/hg38.fa",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None


def resolve_local_mouse_fasta_path():
    candidates = [
        os.getenv("ALPHAGENOME_MOUSE_FASTA"),
        "/data/Dcode/gaetano/repos/fasta_files/mm10.fa",
        "/data/Dcode/gaetano/repos/fasta_files/mm39.fa",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None


ensure_hf_auth(allow_prompt=True)

human_fasta = resolve_local_human_fasta_path()
if not human_fasta:
    raise FileNotFoundError(
        "No local human FASTA found. Set ALPHAGENOME_FASTA to a local GRCh38 FASTA path, e.g.\n"
        "export ALPHAGENOME_FASTA=/path/to/GRCh38.p13.genome.fa"
    )

mouse_fasta = resolve_local_mouse_fasta_path()
if mouse_fasta:
    print(f"Using local mouse FASTA: {mouse_fasta}")
else:
    print(
        "No local mouse FASTA found; configuring mouse organism without local FASTA. "
        "Predictions on human intervals still work."
    )

print(f"Using local human FASTA: {human_fasta}")
# Default GENCODE GTF/PAS/splice-site annotations are required by gene-based scorers (e.g. RNA_SEQ).
human_defaults = dna_model.default_organism_settings()[public_dna_model.Organism.HOMO_SAPIENS]
calibration_path = fsspec.open_local(
    "simplecache::https://storage.googleapis.com/alphagenome/data/hg38/calibration_scores.pb",
    mode="rb",
    simplecache={"cache_storage": str(Path.home() / ".cache" / "alphagenome" / "calibration")},
    https={"client_kwargs": {"trust_env": True}},
)
print(f"Using cached human calibration: {calibration_path}")
organism_settings = {
    public_dna_model.Organism.HOMO_SAPIENS: dataclasses.replace(
        human_defaults, fasta_path=human_fasta, calibration_path=calibration_path
    ),
    public_dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings(
        fasta_path=mouse_fasta,
    ) if mouse_fasta else dna_model.OrganismSettings(),
}

model_device = gpu_devices[0]
print(f"Loading AlphaGenome on {model_device}")
model = dna_model.create_from_huggingface(
    "fold-1", organism_settings=organism_settings, device=model_device
)
print("Model loaded successfully.")

## Info AG output

### Modalities

In [ ]:
[output.name for output in dna_client.OutputType]

### biosample

In [ ]:
metadata = model.output_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate()

metadata.columns

### Get Cell-line or Tissue

In [ ]:
biosample_of_interest = [
    # Core ALS
    "motor neuron",
    "spinal cord",
    "neural cell",
    "neural progenitor cell",
    
    # Neurons
    "neuron",
    
    # Glia
    "astrocyte",
    "astrocyte of the spinal cord",
    "oligodendrocyte precursor cell",
    
    # ALS-FTD axis
    "frontal cortex",
    "temporal lobe",
    "dorsolateral prefrontal cortex",
    "anterior cingulate cortex",
]

# Find matching ontology terms for each biosample
matching_terms = {}
for term in biosample_of_interest:
    matches = metadata[metadata['biosample_name'] == term]['ontology_curie'].unique()
    if len(matches) > 0:
        matching_terms[term] = matches[0]  # Take first match
        print(f"✓ {term:45} → {matches[0]}")
    else:
        print(f"✗ {term:45} → NOT FOUND")


print(f"\nFound {len(matching_terms)} / {len(biosample_of_interest)} biosamples")
print(f"\nOntology terms for prediction: {list(matching_terms.values())}")

## Predictions 

In [ ]:
POSITION = 1000000

variant_seq = genome.Variant(
    chromosome='chr9', 
    position=POSITION, 
    reference_bases='G', 
    alternate_bases='C'
)

print(f"Variant created: {variant_seq.chromosome}:{variant_seq.position}")
print(f"  REF: {variant_seq.reference_bases}")
print(f"  ALT: {variant_seq.alternate_bases[:50]}... ({len(variant_seq.alternate_bases)} bp)")

interval = variant_seq.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

# Define variant scorers to use
selected_scorers = [
    "RNA_SEQ",
    "CHIP_HISTONE_ACTIVE",
    # add more if needed:
    "ATAC_ACTIVE",
    "DNASE_ACTIVE",
    "CHIP_TF_ACTIVE",
]


variant_scores = model.score_variant(
            interval=interval,
            variant=variant_seq,
            variant_scorers=[
                variant_scorers.RECOMMENDED_VARIANT_SCORERS[name]
                for name in selected_scorers
            ],
        )

In [ ]:
missing_quantiles = [
    str(scores.uns["variant_scorer"])
    for scores in variant_scores
    if "quantiles" not in scores.layers
]
if missing_quantiles:
    raise RuntimeError(
        f"Quantile calibration is missing for: {missing_quantiles}. "
        "Restart the kernel and rerun the model setup with human calibration enabled."
    )

df_snp = variant_scorers.tidy_scores([variant_scores], match_gene_strand=True)
if df_snp is None or df_snp.empty:
    raise ValueError("No score rows returned; check the variant, interval, and selected scorers.")

with pd.option_context("display.max_columns", None):
    display(df_snp.head())